In [1]:
from huggingface_hub import login
from transformers import pipeline
from dotenv import load_dotenv
import os
from datasets import load_dataset

In [2]:
load_dotenv()
hf_token = os.getenv('HF_TOKEN')
assert  hf_token.startswith('hf_'), 'HF key has wrong value'
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
path = ('lite_dataset')
dataset = load_dataset("imagefolder",data_dir=path)

Resolving data files:   0%|          | 0/2001 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]

In [5]:
detector = pipeline(model="facebook/detr-resnet-50")

Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

[transformers] DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
result = detector('img/8b19784a38a9fea76c5930ad42f22a23.jpg')

In [7]:
def convert_coord_to_xywh(box, size):
  dx = abs(box[2]-box[0])
  dy = abs(box[3]-box[1])
  x = (box[0] + dx/2)/size[0]
  y = (box[1] + dy/2)/size[1]
  w = dx/size[0]
  h = dy/size[1]
  return [x,y,w,h]

In [8]:
def pipeline_model(img):
  result = detector(img)
  bbox = []
  categories = []
  w,h = img.size
  size = (w,h)
  for item in result:
    x0 = item['box']['xmin']
    y0 = item['box']['ymin']
    x1 = item['box']['xmax']
    y1 = item['box']['ymax']

    box = [x0,y0,x1,y1]
    box_wh = convert_coord_to_xywh(box, size)

    bbox.append(box_wh)
    categories.append(item['label'])

  return {'bbox': bbox, 'categories': categories}


In [9]:
from evaluation_v2 import evaluate_model

In [10]:
evaluate_model(pipeline_model,dataset)

  0%|          | 0/200 [00:00<?, ?it/s]

Pipeline Model results:   <b>IoU:</b> 0.812 , Distribution: 4.5% (IoU(0)), 4.0%  (IoU(mid)), 74.5% (IoU(80))
